# Financial Document Analysis — Data Preprocessing & Analysis

This notebook provides exploratory data analysis and visualization of the preprocessed data used in the multimodal financial forecasting system. It covers:

1. **Price Data Overview** — OHLCV statistics and distributions
2. **Engineered Features** — Returns, volatility measures, moving averages
3. **Target Variables** — Direction labels and realized volatility
4. **Macroeconomic Features** — Market regime indicators
5. **Data Split Analysis** — Train/validation/test distributions
6. **Correlation Analysis** — Feature relationships

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Paths
DATA_DIR = Path('../data')
PRICE_DIR = DATA_DIR / 'raw' / 'prices'
TARGET_DIR = DATA_DIR / 'targets'

print('Data directories:')
print(f'  Prices:  {PRICE_DIR}')
print(f'  Targets: {TARGET_DIR}')

## 1. Price Data Overview

In [ ]:
# Load a sample of tickers
SAMPLE_TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'TSLA', 'META', 'AMD']

price_dfs = {}
for ticker in SAMPLE_TICKERS:
    fp = PRICE_DIR / f'{ticker}_ohlcv.csv'
    if fp.exists():
        df = pd.read_csv(fp, parse_dates=['date'])
        price_dfs[ticker] = df

print(f'Loaded {len(price_dfs)} tickers')
print(f'\nSample data (AAPL):')
price_dfs['AAPL'].head()

In [ ]:
# Summary statistics for all available tickers
all_price_files = list(PRICE_DIR.glob('*_ohlcv.csv'))
print(f'Total price files: {len(all_price_files)}')
print(f'Tickers: {[f.stem.replace("_ohlcv", "") for f in sorted(all_price_files)]}')

# Date range and observation counts
summary = []
for fp in sorted(all_price_files):
    df = pd.read_csv(fp, parse_dates=['date'])
    ticker = fp.stem.replace('_ohlcv', '')
    if ticker == 'tech10':
        continue
    summary.append({
        'Ticker': ticker,
        'Start': df['date'].min().strftime('%Y-%m-%d'),
        'End': df['date'].max().strftime('%Y-%m-%d'),
        'Observations': len(df),
        'Avg Close': df['close'].mean().round(2),
    })

summary_df = pd.DataFrame(summary)
print(f'\nDataset covers {summary_df["Start"].min()} to {summary_df["End"].max()}')
print(f'Avg observations per ticker: {summary_df["Observations"].mean():.0f}')
summary_df

In [ ]:
# Normalized price trajectories
fig, ax = plt.subplots(figsize=(16, 7))

for ticker, df in price_dfs.items():
    normalized = df['close'] / df['close'].iloc[0] * 100
    ax.plot(df['date'], normalized, label=ticker, linewidth=1.2)

ax.set_title('Normalized Price Trajectories (Base = 100)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Normalized Price')
ax.legend(loc='upper left', ncol=4, fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 2. Engineered Features

In [ ]:
# Compute features for AAPL as an example
df = price_dfs['AAPL'].copy()

# Returns
df['log_return'] = np.log(df['close'] / df['close'].shift(1))
df['intraday_return'] = (df['close'] - df['open']) / df['open']

# Moving averages
for w in [5, 10, 20]:
    df[f'sma_{w}'] = df['close'].rolling(w).mean()

# Volatility measures
df['parkinson_vol'] = np.sqrt(
    (1 / (4 * np.log(2))) * (np.log(df['high'] / df['low']))**2
)
df['garman_klass_vol'] = np.sqrt(
    0.5 * (np.log(df['high'] / df['low']))**2
    - (2 * np.log(2) - 1) * (np.log(df['close'] / df['open']))**2
)

# HAR-RV features
df['rv_daily'] = df['log_return']**2
df['rv_weekly'] = df['rv_daily'].rolling(5).mean()
df['rv_monthly'] = df['rv_daily'].rolling(22).mean()

df = df.dropna()
print(f'Engineered features for AAPL: {len(df)} observations')
df[['date', 'close', 'log_return', 'parkinson_vol', 'garman_klass_vol', 'rv_daily', 'rv_weekly', 'rv_monthly']].describe().round(6)

In [ ]:
# Return distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Log returns histogram
axes[0].hist(df['log_return'], bins=80, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_title('AAPL Daily Log Returns', fontweight='bold')
axes[0].set_xlabel('Log Return')
axes[0].set_ylabel('Frequency')
mean_r = df['log_return'].mean()
std_r = df['log_return'].std()
skew_r = df['log_return'].skew()
kurt_r = df['log_return'].kurtosis()
axes[0].text(0.02, 0.95, f'Mean: {mean_r:.5f}\nStd: {std_r:.4f}\nSkew: {skew_r:.2f}\nKurt: {kurt_r:.2f}',
             transform=axes[0].transAxes, fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# QQ plot
from scipy import stats
stats.probplot(df['log_return'].dropna(), dist='norm', plot=axes[1])
axes[1].set_title('QQ Plot vs Normal', fontweight='bold')

# Intraday return distribution
axes[2].hist(df['intraday_return'], bins=80, color='coral', alpha=0.7, edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[2].set_title('AAPL Intraday Returns', fontweight='bold')
axes[2].set_xlabel('Intraday Return')

plt.tight_layout()
plt.show()

In [ ]:
# Moving averages plot
fig, ax = plt.subplots(figsize=(16, 6))

recent = df[df['date'] >= '2023-01-01'].copy()
ax.plot(recent['date'], recent['close'], label='Close', linewidth=1.5, color='black')
for w, color in zip([5, 10, 20], ['#e74c3c', '#3498db', '#2ecc71']):
    ax.plot(recent['date'], recent[f'sma_{w}'], label=f'SMA-{w}', linewidth=1, alpha=0.8, color=color)

ax.set_title('AAPL Close Price with Moving Averages (2023+)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price ($)')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

In [ ]:
# Volatility measures comparison
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(df['date'], df['parkinson_vol'], label='Parkinson', alpha=0.6, linewidth=0.8)
axes[0].plot(df['date'], df['garman_klass_vol'], label='Garman-Klass', alpha=0.6, linewidth=0.8)
axes[0].set_title('Intraday Volatility Estimators', fontweight='bold')
axes[0].set_ylabel('Volatility')
axes[0].legend()

axes[1].plot(df['date'], df['rv_daily'], label='Daily RV', alpha=0.4, linewidth=0.5, color='gray')
axes[1].plot(df['date'], df['rv_weekly'], label='Weekly RV (5d)', alpha=0.8, linewidth=1)
axes[1].plot(df['date'], df['rv_monthly'], label='Monthly RV (22d)', alpha=0.8, linewidth=1.2)
axes[1].set_title('HAR-RV Components', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Realized Variance')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Target Variables

In [ ]:
# Load direction labels
dir_labels = pd.read_csv(TARGET_DIR / 'direction_labels_multi_horizon.csv', parse_dates=['date'])
print(f'Direction labels: {len(dir_labels)} rows, {dir_labels["ticker"].nunique()} tickers')
print(f'Date range: {dir_labels["date"].min()} to {dir_labels["date"].max()}')
dir_labels.head()

In [ ]:
# Direction label distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, horizon in enumerate(['5d', '60d']):
    col = f'direction_{horizon}_label'
    counts = dir_labels[col].value_counts()
    colors = ['#2ecc71', '#e74c3c']
    axes[i].bar(counts.index, counts.values, color=colors, edgecolor='white', width=0.5)
    axes[i].set_title(f'{horizon.upper()} Direction Label Distribution', fontweight='bold')
    axes[i].set_ylabel('Count')
    total = counts.sum()
    for j, (label, count) in enumerate(counts.items()):
        axes[i].text(j, count + total * 0.01, f'{count/total:.1%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Direction label distribution per ticker
fig, ax = plt.subplots(figsize=(16, 6))

ticker_dir = dir_labels.groupby('ticker')['direction_60d_label'].value_counts().unstack(fill_value=0)
ticker_dir_pct = ticker_dir.div(ticker_dir.sum(axis=1), axis=0)

ticker_dir_pct.plot(kind='bar', stacked=True, ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='white')
ax.set_title('60-Day Direction Label Proportions by Ticker', fontweight='bold')
ax.set_ylabel('Proportion')
ax.set_xlabel('Ticker')
ax.axhline(0.5, color='black', linestyle='--', alpha=0.5)
ax.legend(title='Direction', loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Forward return distributions
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, horizon in enumerate(['5d', '60d']):
    col = f'direction_{horizon}_return'
    data = dir_labels[col].dropna()
    axes[i].hist(data, bins=100, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].axvline(0, color='red', linestyle='--', linewidth=1.5)
    axes[i].set_title(f'{horizon.upper()} Forward Return Distribution', fontweight='bold')
    axes[i].set_xlabel('Return')
    axes[i].set_ylabel('Frequency')
    axes[i].text(0.02, 0.95, f'Mean: {data.mean():.4f}\nStd: {data.std():.4f}\nMedian: {data.median():.4f}',
                 transform=axes[i].transAxes, fontsize=9, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
# Load volatility targets
vol_targets = pd.read_csv(TARGET_DIR / 'volatility_targets.csv', parse_dates=['date'])
print(f'Volatility targets: {len(vol_targets)} rows, {vol_targets["ticker"].nunique()} tickers')
vol_targets.head()

In [ ]:
# Volatility distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

vol_data = vol_targets['realized_vol_20d_annualized'].dropna()

axes[0].hist(vol_data, bins=100, color='darkorange', alpha=0.7, edgecolor='white')
axes[0].set_title('Annualized 20-Day Realized Volatility Distribution', fontweight='bold')
axes[0].set_xlabel('Annualized Volatility')
axes[0].set_ylabel('Frequency')
axes[0].text(0.65, 0.95, f'Mean: {vol_data.mean():.4f}\nMedian: {vol_data.median():.4f}\nStd: {vol_data.std():.4f}',
             transform=axes[0].transAxes, fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Per-ticker volatility boxplot
ticker_vol = vol_targets.groupby('ticker')['realized_vol_20d_annualized'].apply(list).to_dict()
tickers_sorted = sorted(ticker_vol.keys())
axes[1].boxplot([ticker_vol[t] for t in tickers_sorted], labels=tickers_sorted,
                patch_artist=True, showfliers=False,
                boxprops=dict(facecolor='lightskyblue', alpha=0.7))
axes[1].set_title('Realized Volatility by Ticker', fontweight='bold')
axes[1].set_ylabel('Annualized Volatility')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Volatility time series for select tickers
fig, ax = plt.subplots(figsize=(16, 6))

for ticker in ['AAPL', 'NVDA', 'TSLA', 'MSFT']:
    subset = vol_targets[vol_targets['ticker'] == ticker]
    ax.plot(subset['date'], subset['realized_vol_20d_annualized'],
            label=ticker, linewidth=0.8, alpha=0.8)

ax.set_title('Annualized Realized Volatility Over Time', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

## 4. Data Split Analysis

In [ ]:
# Demonstrate time-based splitting
def time_based_split(df, train_ratio=0.7, val_ratio=0.15):
    df = df.sort_values('date').reset_index(drop=True)
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))
    return df.iloc[:train_end], df.iloc[train_end:val_end], df.iloc[val_end:]

aapl_vol = vol_targets[vol_targets['ticker'] == 'AAPL'].copy()
train, val, test = time_based_split(aapl_vol)

print('AAPL Time-Based Split:')
print(f'  Train: {train["date"].min().strftime("%Y-%m-%d")} to {train["date"].max().strftime("%Y-%m-%d")} ({len(train)} obs)')
print(f'  Val:   {val["date"].min().strftime("%Y-%m-%d")} to {val["date"].max().strftime("%Y-%m-%d")} ({len(val)} obs)')
print(f'  Test:  {test["date"].min().strftime("%Y-%m-%d")} to {test["date"].max().strftime("%Y-%m-%d")} ({len(test)} obs)')

# Visualize the split
fig, ax = plt.subplots(figsize=(16, 5))

ax.plot(train['date'], train['realized_vol_20d_annualized'], color='#3498db', label='Train', linewidth=0.8)
ax.plot(val['date'], val['realized_vol_20d_annualized'], color='#f39c12', label='Validation', linewidth=0.8)
ax.plot(test['date'], test['realized_vol_20d_annualized'], color='#e74c3c', label='Test', linewidth=0.8)

ax.axvline(val['date'].iloc[0], color='gray', linestyle='--', alpha=0.7)
ax.axvline(test['date'].iloc[0], color='gray', linestyle='--', alpha=0.7)

ax.set_title('AAPL — Chronological Train / Validation / Test Split', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

In [ ]:
# Compare distributions across splits
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (name, split, color) in enumerate([
    ('Train', train, '#3498db'),
    ('Validation', val, '#f39c12'),
    ('Test', test, '#e74c3c'),
]):
    vol = split['realized_vol_20d_annualized']
    ret = split['log_return']
    
    axes[i].hist(vol, bins=50, color=color, alpha=0.7, edgecolor='white')
    axes[i].set_title(f'{name} Set — Volatility Distribution', fontweight='bold')
    axes[i].set_xlabel('Annualized Volatility')
    axes[i].set_ylabel('Frequency')
    axes[i].text(0.6, 0.95,
                 f'N: {len(split)}\nMean: {vol.mean():.3f}\nStd: {vol.std():.3f}',
                 transform=axes[i].transAxes, fontsize=9, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Cross-ticker return correlations
pivot_returns = pd.DataFrame()
for ticker, pdf in price_dfs.items():
    pdf = pdf.set_index('date')
    pivot_returns[ticker] = np.log(pdf['close'] / pdf['close'].shift(1))

corr_matrix = pivot_returns.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Cross-Ticker Log Return Correlations', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap for AAPL
feature_cols = ['log_return', 'intraday_return', 'parkinson_vol', 'garman_klass_vol',
                'rv_daily', 'rv_weekly', 'rv_monthly']
corr_features = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr_features, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('AAPL — Feature Correlations', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Z-Score Normalization Demonstration

In [ ]:
# Demonstrate z-score normalization using train-only statistics
feature_cols_norm = ['log_return', 'parkinson_vol', 'garman_klass_vol', 'rv_weekly']

train_feat = train.merge(df[['date'] + feature_cols_norm], on='date', how='inner')
val_feat = val.merge(df[['date'] + feature_cols_norm], on='date', how='inner')
test_feat = test.merge(df[['date'] + feature_cols_norm], on='date', how='inner')

# Fit on train only
means = train_feat[feature_cols_norm].mean()
stds = train_feat[feature_cols_norm].std().replace(0, 1)

print('Training set statistics (used for normalization):')
stats_table = pd.DataFrame({'Mean': means, 'Std': stds})
print(stats_table.round(6))

# Apply normalization
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Before normalization
for col in feature_cols_norm:
    axes[0].hist(train_feat[col], bins=50, alpha=0.5, label=col)
axes[0].set_title('Before Z-Score Normalization', fontweight='bold')
axes[0].set_xlabel('Raw Feature Value')
axes[0].legend(fontsize=8)

# After normalization
train_norm = (train_feat[feature_cols_norm] - means) / stds
for col in feature_cols_norm:
    axes[1].hist(train_norm[col], bins=50, alpha=0.5, label=col)
axes[1].set_title('After Z-Score Normalization', fontweight='bold')
axes[1].set_xlabel('Normalized Value')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Fundamental Surprise Analysis

In [ ]:
# Load fundamental surprise targets
surprise_fp = TARGET_DIR / 'fundamental_surprise_targets.csv'
if surprise_fp.exists():
    surprise_df = pd.read_csv(surprise_fp, parse_dates=['date'])
    print(f'Fundamental surprise targets: {len(surprise_df)} rows')
    print(f'Columns: {list(surprise_df.columns)}')
    surprise_df.head(10)
else:
    print('Fundamental surprise target file not found.')

## 8. Summary Statistics

In [ ]:
# Final summary
print('='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'\nCompany Universe:    30 S&P 500 tech stocks')
print(f'Price Data:          ~{summary_df["Observations"].mean():.0f} trading days per ticker')
print(f'Direction Labels:    {len(dir_labels):,} total observations')
print(f'Volatility Targets:  {len(vol_targets):,} total observations')
print(f'\nTarget Variables:')
print(f'  Direction (5d):    {dir_labels["direction_5d_label"].value_counts().to_dict()}')
print(f'  Direction (60d):   {dir_labels["direction_60d_label"].value_counts().to_dict()}')
print(f'  Volatility:        Mean={vol_targets["realized_vol_20d_annualized"].mean():.4f}, Std={vol_targets["realized_vol_20d_annualized"].std():.4f}')
print(f'\nEngineered Features: 21 price features per window')
print(f'Macro Features:      12 market indicators (1d lag)')
print(f'Text Features:       FinBERT 768-dim embeddings')
print(f'Graph Features:      GAT 32-dim embeddings')
print(f'\nData Splits:         70% train / 15% val / 15% test')
print(f'Normalization:       Z-score (train-fitted)')
print('='*60)